# 市场汇总导入 MySQL

将按国家汇总的交易指标导入 lavel2_country_copy 表。空库复现的表名衔接见[项目说明](../README.md)。

In [1]:
import pandas as pd
from sqlalchemy import URL, create_engine, text

from mysql_import import write_dataframe_to_mysql

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / "../config/.env", interpolate=False, encoding="utf-8-sig")
mysql_user = os.getenv("UCI_MYSQL_USER")
mysql_password = os.getenv("UCI_MYSQL_PASSWORD")
if not mysql_user or not mysql_password:
    raise RuntimeError("Set UCI_MYSQL_USER and UCI_MYSQL_PASSWORD in the kernel environment or in ../config/.env.")

# 创建数据库连接地址
url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="127.0.0.1",
    port=3306,
    database="data_10month",
    query={"charset": "utf8mb4"}
)

# 创建连接引擎
engine = create_engine(url, pool_pre_ping=True)

# 检查连接的数据库和 MySQL 版本
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT DATABASE(), VERSION()")
    )
    print(result.fetchone())

('data_10month', '8.4.11')


In [3]:
df = pd.read_csv("../data/outputs/data10_country.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   country           26 non-null     object 
 1   gross_amount_gbp  26 non-null     float64
 2   invoice_count     26 non-null     int64  
 3   avg_invoice_gbp   26 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 964.0+ bytes


In [4]:
write_dataframe_to_mysql(df, "lavel2_country_copy", engine)

26